# Enunciado práctica.

### Recursos

Problemas interesantes para Aprendizaje por refuerzo
 * Gymnasium: https://gymnasium.farama.org/environments/box2d/
 * Solución del Lunar Lander con DQN: https://shiva-verma.medium.com/solving-lunar-lander-openaigym-reinforcement-learning-785675066197
 * Otra solución: https://wingedsheep.com/lunar-lander-dqn/ y https://github.com/wingedsheep/blog/blob/main/lunar-lander-dqn/lunar_lander_dqn_blog.ipynb
 * The Nature of Code: https://youtu.be/lu5ul7z4icQ
 * Librería para neuroevolución: https://pypi.org/project/nevopy/

In [ ]:
!pip install -r ../../requirements.txt --quiet

In [ ]:
import gymnasium as gym
import numpy as np
import pygame
import gymnasium.utils.play


In [7]:

class EVO_Handler:
    def __init__(self, model, environment, population_size=100):
        self.model = model
        self.env = environment

        self.population_size = population_size

        def generate_population():
            len_chromosome = len(self.model.to_chromosome())
            return np.random.uniform(-1, 1, (self.population_size, len_chromosome))

        self.population = generate_population()

    def _evaluate_population(self, reward_function=None):
        fitness = np.zeros(self.population_size)

        N = 3
        for i in range(self.population_size):
            ch = self.population[i]
            self.model.from_chromosome(ch)

            for _ in range(N): # media de 5 episodios para cada individuo

                observation, info = self.env.reset()
                terminated = False
                truncated = False

                max_steps = 1000
                steps = 0

                while not (terminated or truncated) and steps < max_steps:
                    action = self.model.forward(observation)

                    observation, reward, terminated, truncated, info = self.env.step(action)

                    steps += 1

                    if reward_function:
                        fitness[i] += reward_function(observation, reward)
                    else:
                        fitness[i] += reward

                if steps >= max_steps:
                    fitness[i] -= 100 # Penalización por no terminar el episodio


        return fitness / N

    def sort_population(self, fitness):
        sorted_indices = np.argsort(fitness)[::-1]
        self.population = self.population[sorted_indices]
        self.fitness = fitness[sorted_indices]
        return sorted_indices


    def evolve(self, selection, crossover, mutation, generations=100, reward_function=None, elitism_percentage=0.3):

        best_fitness = -np.inf
        best_chromosome = None

        for gen in range(generations):
            fitness = self._evaluate_population(reward_function)
            sorted_indices = self.sort_population(fitness)

            if self.fitness[0] > best_fitness:
                best_fitness = self.fitness[0]
                best_chromosome = self.population[0]

            if gen % 10 == 0:
                print(f"Generation {gen} - Best fitness: {self.fitness[0]} - Average fitness: {np.mean(self.fitness)}")

            new_gen = []

            num_elite = int(self.population_size * elitism_percentage)
            new_gen.extend(self.population[:num_elite])

            while len(new_gen) < self.population_size:


                parent1 = selection(self.population, self.fitness)
                parent2 = selection(self.population, self.fitness)
                child = crossover(parent1, parent2)
                child = mutation(child)
                new_gen.append(child)

            self.population = np.array(new_gen)

        return best_chromosome, best_fitness

In [8]:
def tournament_selection(population, fitness):

    K = 3

    selected_indices = np.random.choice(len(population), K, replace=False)
    selected_fitness = fitness[selected_indices]
    winner_index = selected_indices[np.argmax(selected_fitness)]
    return population[winner_index]




def blx_alpha_crossover(parent1, parent2):

    alpha = 0.5
    child = np.zeros_like(parent1)

    for i in range(len(parent1)):
        c_min = min(parent1[i], parent2[i])
        c_max = max(parent1[i], parent2[i])
        I = c_max - c_min

        lower_bound = c_min - alpha * I
        upper_bound = c_max + alpha * I

        child[i] = np.random.uniform(lower_bound, upper_bound)

    return child

def uniform_crossover(parent1, parent2):

    p = np.random.uniform(0, 1)

    if p < 0.5:
        return parent1.copy()
    else:
        return parent2.copy()



def gaussian_mutation(chromosome, mutation_rate=0.4):
    sigma = 0.1

    new_chromosome = chromosome.copy()
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            new_chromosome[i] += np.random.normal(0, sigma)
            # reset esporádico
            if np.random.rand() < 0.02: # 2% de probabilidad de reset
                new_chromosome[i] = np.random.uniform(-1, 1)
    return new_chromosome

## Bipedal Walker

Para el Bipedal Walker nuestro espacio de observación no solo tiene 8 números, sino que este consta de 24:
- El ángulo y la velocidad angular del torso, 2 números
- Velocidad horizontal y vertical, 2 números
- Ángulo y velocidad de cada una de sus 4 extremidades (caderas/rodilla), 8 números
- Si los pies están tocando el suelo, 2 números
- Radares "lidar" para comprobar el desnivel del teerreno, 10 números.

Además, aquí podemos usar las 4 extremidades a la vez y no tenemos que elegir solo una de las posibles opciones como en el lunar lander. Aquí movemos las 4 articulaciones a la vez.

El espacio de acciones es un array de la fuerza hacia adelante o hacia atrás que le damos a  cada una de las extremidades, en un rango de -1 a 1.


Para la creación de nuestro modelo (nuestra red neuronal) cambiaremos dos cosas:
- Como la entrada es de 24 neuronas cambiaremos la arquitectura a 24->16->8->4. Usamos más neuronas en las capas ocultas porque es un problema más complejo y con más parámetros, así puede aprender mejor.
- Además no usaremos la softmax ya que no solo vamos a elegir la mejor acción. En vez de esta, haremos una función tanh, ya que convierte los outputs al rango (-1, 1), que es justo el rango de los outputs posibles de cada una de las extremidades.


Esto lo hacemos en Gymnasium_bipedal.py porque le haremos paralelismo y la librería de paralelismo funciona mejor en .py.

In [9]:
import numpy as np
from Gymnasium_bipedal import BipedalWalker, evaluate_chromosome_parallel

bipedal_walker = BipedalWalker()

Para entrenarlo usaremos la misma clase de EVO_Handler que ya teníamos creada para el Lunar Lander.

La función de recompensa por defecto de Gymnasium es la siguiente:
- Da puntos para por moverse hacia adelante, llegando hasta 300 puntos si llega al final de todo.
- Si el robot muere (se cae), pierde 100 puntos.
- Además, lo más interesante es que también resta puntos proporcional a la cantidad de fuerza que le aplica a cada extremidad, haciendo que la evolución busque un Walker que ande de manera fluida y con pocos tirones o movimientos violentos.

Como esta función creemos que ya tiene muy buena pinta, la usaremos para entrenar el modelo.



Empezamos creando el entorno.


In [10]:
env_bipedal_walker = gym.make("BipedalWalker-v3")

In [11]:
import concurrent.futures

class EVO_Handler_Paralelo(EVO_Handler): 
    def _evaluate_population(self, reward_function=None):
        with concurrent.futures.ProcessPoolExecutor() as executor:
            resultados_fitness = list(executor.map(evaluate_chromosome_parallel, self.population))

        return np.array(resultados_fitness)

In [12]:
handler_bipedal_walker = EVO_Handler_Paralelo(
    model=bipedal_walker, 
    environment=env_bipedal_walker, 
    population_size=100)

Para el la selección seguiremos usando la de torneo, sin embargo vamos a cambiar la mutación y el cruce.

Para el cruce usaremos el BLX-alpha porque al fin y al cabo el uniform realmente es clonar uno de los dos padres. Para que haya un cruce real hay que usar el blx alpha, que además está hecho para número continuos como son los pesos de la red neurnoal. Lo que hace este cruce es mirar en el espacio entre los dos padres y además un poquito más allá de los extremos, lo que lo hace muy bueno para la exploración.

Para la mutación bajaremos el porcentaje de genes que mutamos a solamente un 5-10% ya que si ponemos más sería un número demasiado grande y cambiaría prácticamente el individuo completamente, en un problema tan complejo podría haber hecho que le sea mucho más complicado aprender.

In [13]:
def gaussian_mutation_bipedal_walker(chromosome, mutation_rate=0.05):
    sigma = 0.1 # cantidad cambio que le sumamos

    new_chromosome = chromosome.copy()
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            new_chromosome[i] += np.random.normal(0, sigma)

            #0.5% de posibilidades de resetear el individuo completamente, por si
            # todos los individuos se han quedado atascados en el mismo mínimo salir.
            if np.random.rand() < 0.005:
                new_chromosome[i] = np.random.uniform(-1, 1)

    return new_chromosome

In [14]:
if __name__ == '__main__':
    
    best_walker_chromosome, best_walker_fitness = handler_bipedal_walker.evolve(
        tournament_selection,
        blx_alpha_crossover,
        gaussian_mutation_bipedal_walker,
        generations=2000
)
    
    print(f"\nMejor puntuación: {best_walker_fitness}")

Generation 0 - Best fitness: -90.7322769165039 - Average fitness: -141.80857849121094
Generation 10 - Best fitness: -96.27852630615234 - Average fitness: -115.06404113769531
Generation 20 - Best fitness: -95.6875 - Average fitness: -110.99884033203125
Generation 30 - Best fitness: -94.4011459350586 - Average fitness: -105.91845703125
Generation 40 - Best fitness: -92.63507080078125 - Average fitness: -100.68787384033203
Generation 50 - Best fitness: -91.62032318115234 - Average fitness: -94.15533447265625
Generation 60 - Best fitness: -89.2890396118164 - Average fitness: -95.56585693359375
Generation 70 - Best fitness: -90.47566986083984 - Average fitness: -94.35162353515625
Generation 80 - Best fitness: -89.06485748291016 - Average fitness: -94.89804077148438
Generation 90 - Best fitness: -86.30681610107422 - Average fitness: -94.53399658203125
Generation 100 - Best fitness: -81.62345886230469 - Average fitness: -91.71125030517578
Generation 110 - Best fitness: -83.15837097167969 - Av

KeyboardInterrupt: 

Parece que se queda estancado en aprox (-87,-88). Puede que se queda siempre en el mismo local o que realmente el problema es muy complicado y está aprendiendo a ponerse de pie o a andar, solo que necesita muchas más epochs.

Vamos a probar a subir el porcentaje de mutaciones para que tenga un mayor espacio de exploración y además para que no se estanquen todos los robots en un local vamos a subir la probabilidad de reseteo y el número de población.

In [17]:
def gaussian_mutation_bipedal_walker(chromosome, mutation_rate=0.15): # Subimos al 15%
    sigma = 0.1 

    new_chromosome = chromosome.copy()
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate: 
            new_chromosome[i] += np.random.normal(0, sigma)

            # Subimos el reseteo al 2%
            if np.random.rand() < 0.02: 
                new_chromosome[i] = np.random.uniform(-1, 1)

    return new_chromosome

handler_bipedal_walker = EVO_Handler_Paralelo(
    model=bipedal_walker, 
    environment=env_bipedal_walker, 
    population_size=200)

In [18]:
if __name__ == '__main__':
    
    best_walker_chromosome, best_walker_fitness = handler_bipedal_walker.evolve(
        tournament_selection,
        blx_alpha_crossover,
        gaussian_mutation_bipedal_walker,
        generations=2000
)
    
    print(f"\nMejor puntuación: {best_walker_fitness}")

Generation 0 - Best fitness: -97.6259765625 - Average fitness: -139.6556396484375
Generation 10 - Best fitness: -97.0127182006836 - Average fitness: -120.05028533935547
Generation 20 - Best fitness: -93.8438949584961 - Average fitness: -114.8851547241211
Generation 30 - Best fitness: -93.95487213134766 - Average fitness: -119.69218444824219
Generation 40 - Best fitness: -89.10411834716797 - Average fitness: -109.37800598144531
Generation 50 - Best fitness: -91.85595703125 - Average fitness: -112.01652526855469
Generation 60 - Best fitness: -91.8971176147461 - Average fitness: -105.49124145507812
Generation 70 - Best fitness: -91.56201171875 - Average fitness: -103.68402099609375
Generation 80 - Best fitness: -90.9054946899414 - Average fitness: -102.23333740234375
Generation 90 - Best fitness: -88.21075439453125 - Average fitness: -102.36077117919922
Generation 100 - Best fitness: -84.27757263183594 - Average fitness: -100.4786148071289
Generation 110 - Best fitness: -90.2723159790039 

Exception ignored in: <function b2JointDef.__del__ at 0x7f4a17e59b20>
Traceback (most recent call last):
  File "/home/jordi/Desktop/Tercero/Linux_segundo_cuatri/Sistemas Inteligentes/neuro_evo/.venv/lib/python3.13/site-packages/Box2D/Box2D.py", line 3588, in __del__
    self.ClearUserData()
TypeError: in method 'b2JointDef_ClearUserData', argument 1 of type 'b2JointDef *'


KeyboardInterrupt: 

Vemos que tampoco es de mucha ayuda, vamos a hacer una última prueba bajándole el elitismo para que no se quede a tanto porcentaje de buenos y vamos a modificar el BipedalWalker para que la red neuronal pueda captar mayor complejidad, ya que quizás 24 neuronas de entrada necesita una arquitectura más complicada.

In [19]:
class BipedalWalker(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(24, 40), 
            nn.ReLU(),
            nn.Linear(40, 40),
            nn.ReLU(),
            nn.Linear(40, 4)
        )

    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32)

        with torch.no_grad():
            x = self.fc(x)
            x = torch.tanh(x)
            
        return x.numpy()
    
    def from_chromosome(self, chromosome):
        vector_to_parameters(torch.tensor(chromosome, dtype=torch.float32), self.parameters())
    
    def to_chromosome(self):
        return parameters_to_vector(self.parameters()).detach().numpy()


In [20]:
bipedal_walker = BipedalWalker()

In [ ]:
handler_bipedal_walker_2 = EVO_Handler_Paralelo(
    model=bipedal_walker, 
    environment=env_bipedal_walker, 
    population_size=150)

In [23]:
if __name__ == '__main__':
    
    best_walker_chromosome_2, best_walker_fitness_2 = handler_bipedal_walker_2.evolve(
        tournament_selection,
        blx_alpha_crossover,
        gaussian_mutation_bipedal_walker,
        generations=2000,
        elitism_percentage=0.05
    )
    
    print(f"\nMejor puntuación: {best_walker_fitness_2}")

Generation 0 - Best fitness: -97.2357177734375 - Average fitness: -138.0054473876953
Generation 10 - Best fitness: -96.1343765258789 - Average fitness: -129.10801696777344
Generation 20 - Best fitness: -93.6260986328125 - Average fitness: -114.81455993652344
Generation 30 - Best fitness: -93.6487045288086 - Average fitness: -117.75952911376953
Generation 40 - Best fitness: -93.20076751708984 - Average fitness: -123.39347839355469
Generation 50 - Best fitness: -92.4727783203125 - Average fitness: -118.6738510131836
Generation 60 - Best fitness: -92.20144653320312 - Average fitness: -111.97740936279297
Generation 70 - Best fitness: -91.5792007446289 - Average fitness: -114.82427215576172
Generation 80 - Best fitness: -90.64339447021484 - Average fitness: -109.03267669677734
Generation 90 - Best fitness: -90.24468994140625 - Average fitness: -104.62142944335938
Generation 100 - Best fitness: -88.9879150390625 - Average fitness: -104.42587280273438
Generation 110 - Best fitness: -89.195335

Process ForkProcess-35005:
Process ForkProcess-35004:
Process ForkProcess-35001:
Process ForkProcess-34998:
Process ForkProcess-34999:
Process ForkProcess-34997:
Process ForkProcess-35003:
Process ForkProcess-35002:
Process ForkProcess-34996:
Process ForkProcess-34994:
Process ForkProcess-34993:
Process ForkProcess-35008:
Process ForkProcess-34995:
Process ForkProcess-35007:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/jordi/.local/share/uv/python/cpython-3.13.12-linux-x86_64-gnu/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~

KeyboardInterrupt: 

  File "/home/jordi/.local/share/uv/python/cpython-3.13.12-linux-x86_64-gnu/lib/python3.13/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ~~~~~~~~~~~~~~~~~~~~~~~^^
KeyboardInterrupt


Tras horas de ejecución vemos que los 3 modelos escupen resultados parecidos, por lo que creemos que simplemente necesita muchas epochs y tarda demasiado en aprender.